In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

RUN_MODELS = ["distilbert-base-uncased", "bert-base-uncased"]
RUN_TASKS = ["sst2", "cola", "mrpc"]
SKIP = []                          # (model, task) pairs to leave out, e.g. [("distilbert-base-uncased", "sst2")]
SEEDS = None                          # None = all checkpoints found, or e.g. [0, 1, 2]
N_LIST = [8, 16, 32, 64, 128, 256]    # numbers of dev examples (capped at the dev-set size)
N_DRAWS = 5                           # random subsets per n
K_LIST = [6, 4, 3, 2]                 # heads kept per layer
N_RANDOM = 5
REF_N = 512
OUT_DIR = "/kaggle/working/gradsamples_results"
SMOKE = False                         # True: 1 checkpoint, N=[8,32], 1 draw, K=[4]
SEARCH = [
    "/kaggle/input/notebooks/*/*/ckpt/*/done.json",
    "/kaggle/input/*/ckpt/*/done.json",
    "/kaggle/input/**/done.json",
    "/kaggle/working/ckpt/**/done.json",
]
MAX_LEN = 128

import glob, json, time
import numpy as np
import pandas as pd
from scipy.stats import kendalltau

if SMOKE:
    N_LIST, N_DRAWS, K_LIST, N_RANDOM, OUT_DIR = [8, 32], 1, [4], 1, OUT_DIR + "_smoke"


def uniform_keep(score, k):
    L, H = score.shape
    keep = np.zeros((L, H), bool)
    for l in range(L):
        keep[l, np.argsort(-score[l], kind="stable")[:k]] = True
    return keep


def mean_layer_tau(a, b):
    """Mean over layers of Kendall tau between two (L,H) score arrays."""
    return float(np.mean([kendalltau(a[l], b[l])[0] for l in range(a.shape[0])]))


def overlap(keep_a, keep_b):
    return float((keep_a & keep_b).sum() / max(keep_a.sum(), 1))


import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import matthews_corrcoef, f1_score

dev_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KEYS = {"sst2": ("sentence", None), "cola": ("sentence", None), "mrpc": ("sentence1", "sentence2")}


def get_attn(model):
    if hasattr(model, "distilbert"):
        ls = model.distilbert.transformer.layer
        return [(l.attention.q_lin, l.attention.k_lin, l.attention.v_lin, l.attention.out_lin) for l in ls], \
            model.config.n_heads, model.config.dim
    ls = model.bert.encoder.layer
    return [(l.attention.self.query, l.attention.self.key, l.attention.self.value, l.attention.output.dense) for l in ls], \
        model.config.num_attention_heads, model.config.hidden_size


def prep(task, tok):
    ds = load_dataset("nyu-mll/glue", task)
    k1, k2 = KEYS[task]

    def f(b):
        return tok(b[k1], b[k2], truncation=True, max_length=MAX_LEN) if k2 else tok(b[k1], truncation=True, max_length=MAX_LEN)

    split = ds["train"].train_test_split(test_size=0.05, seed=0)           # identical to fine-tuning script
    parts = {"dev": split["test"], "val": ds["validation"]}
    cols = ["input_ids", "attention_mask", "label"]
    out = {}
    for n, d in parts.items():
        d = d.map(f, batched=True)
        out[n] = d.remove_columns([c for c in d.column_names if c not in cols])
    out["dev_ref"] = out["dev"].select(range(min(REF_N, len(out["dev"]))))
    return out


def make_collate(tok):
    def collate(batch):
        labels = torch.tensor([b["label"] for b in batch])
        enc = tok.pad([{k: b[k] for k in ("input_ids", "attention_mask")} for b in batch], return_tensors="pt")
        enc["labels"] = labels
        return enc
    return collate


@torch.no_grad()
def predict(model, ds, collate, bs=128):
    model.eval()
    lg, lb = [], []
    for b in DataLoader(ds, batch_size=bs, collate_fn=collate):
        lab = b.pop("labels")
        b = {k: v.to(dev_) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev_.type == "cuda"):
            lg.append(model(**b).logits.float().cpu())
        lb.append(lab)
    return torch.cat(lg).numpy(), torch.cat(lb).numpy()


def metrics(task, logits, labels):
    p = logits.argmax(1)
    loss = torch.nn.functional.cross_entropy(torch.tensor(logits), torch.tensor(labels)).item()
    m = {"acc": float((p == labels).mean()), "loss": loss, "mcc": np.nan, "f1": np.nan}
    if task == "cola": m["mcc"] = float(matthews_corrcoef(labels, p))
    if task == "mrpc": m["f1"] = float(f1_score(labels, p, zero_division=0))
    m["primary"] = {"sst2": m["acc"], "cola": m["mcc"], "mrpc": m["f1"]}[task]
    return m


def weight_scores(model):
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    S = {k: np.zeros((L, H)) for k in ["frob", "op", "chain"]}
    with torch.no_grad():
        for l, (q, k, v, o) in enumerate(acc):
            Wq, Wk, Wv, Wo = [m.weight.detach().float().cpu() for m in (q, k, v, o)]
            for h in range(H):
                sl = slice(h * dh, (h + 1) * dh)
                Woh = Wo[:, sl]
                sv = torch.linalg.svdvals(Woh)
                S["op"][l, h] = sv[0].item()
                S["frob"][l, h] = torch.linalg.matrix_norm(Woh, "fro").item()
                bq = torch.linalg.matrix_norm(Wq[sl, :], 2).item()
                bk = torch.linalg.matrix_norm(Wk[sl, :], 2).item()
                bv = torch.linalg.matrix_norm(Wv[sl, :], 2).item()
                S["chain"][l, h] = bq * bk * bv * sv[0].item()
    return S


def grad_importance(model, ds, collate):
    """Michel et al. (2019)-style: mean |dL/dxi_h|, batch size 1, needs labels."""
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    xi = torch.ones(L, H, device=dev_, requires_grad=True)
    hooks = []
    for l, (_, _, _, o) in enumerate(acc):
        def pre(mod, args, l=l):
            x = args[0]
            B, T, Dm = x.shape
            return ((x.reshape(B, T, H, dh) * xi[l].view(1, 1, H, 1)).reshape(B, T, Dm),)
        hooks.append(o.register_forward_pre_hook(pre))
    imp = torch.zeros(L, H, device=dev_)
    model.eval()
    t0 = time.time()
    for b in DataLoader(ds, batch_size=1, collate_fn=collate):
        b = {k: v.to(dev_) for k, v in b.items()}
        loss = model(**b).loss
        g, = torch.autograd.grad(loss, xi)
        imp += g.abs()
    for h_ in hooks:
        h_.remove()
    return (imp / len(ds)).detach().cpu().numpy(), time.time() - t0


@torch.no_grad()
def act_importance(model, ds, collate, bs=32):
    """Label-free, forward-only: mean over valid tokens of ||W_O,h c_h||_2, c_h = head context vector.
    Uses ||W c||^2 = c^T (W^T W) c with the per-head Gram matrix, so no (B,T,H,D) tensor is built."""
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    grams = []
    for (_, _, _, o) in acc:
        Wo = o.weight.detach().float().reshape(D, H, dh)                 # (D, H, dh): columns of head h
        grams.append(torch.einsum("ohd,ohe->hde", Wo, Wo))               # (H, dh, dh)
    sums = torch.zeros(L, H, device=dev_)
    state = {"mask": None, "n": 0}
    hooks = []
    for l, (_, _, _, o) in enumerate(acc):
        def pre(mod, args, l=l):
            x = args[0].float().reshape(args[0].shape[0], args[0].shape[1], H, dh)
            q = torch.einsum("bthd,hde,bthe->bth", x, grams[l], x).clamp_min(0).sqrt()      # (B,T,H)
            sums[l] += (q * state["mask"].unsqueeze(-1)).sum((0, 1))
        hooks.append(o.register_forward_pre_hook(pre))
    model.eval()
    t0 = time.time()
    for b in DataLoader(ds, batch_size=bs, collate_fn=collate):
        b.pop("labels")                                                  # labels are NOT used
        b = {k: v.to(dev_) for k, v in b.items()}
        state["mask"] = b["attention_mask"].float()
        state["n"] += int(b["attention_mask"].sum().item())
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev_.type == "cuda"):
            model(**b)
    for h_ in hooks:
        h_.remove()
    return (sums / max(state["n"], 1)).cpu().numpy(), time.time() - t0


class Pruner:
    def __init__(self, model):
        self.model = model
        self.acc, self.H, self.D = get_attn(model)
        self.dh = self.D // self.H
        self.W0 = [o.weight.detach().clone() for (_, _, _, o) in self.acc]

    def apply(self, keep):
        with torch.no_grad():
            for l, (_, _, _, o) in enumerate(self.acc):
                o.weight.copy_(self.W0[l])
                for h in range(self.H):
                    if not keep[l, h]:
                        o.weight[:, h * self.dh:(h + 1) * self.dh] = 0.0


def discover():
    runs = {}
    for pat in SEARCH:
        for p in glob.glob(pat, recursive=True):
            try:
                r = json.load(open(p))
            except Exception:
                continue
            runs[(r["model"], r["task"], r["seed"])] = os.path.dirname(p)
    sel = {k: v for k, v in runs.items() if k[0] in RUN_MODELS and k[1] in RUN_TASKS and (SEEDS is None or k[2] in SEEDS) and (k[0], k[1]) not in SKIP}
    if not sel:
        print("No matching checkpoints. done.json files seen:", sorted(runs.keys()))
    sel = dict(sorted(sel.items()))
    if SMOKE:
        sel = dict(list(sel.items())[:1])
    return sel


def run_checkpoint(key, path, cache):
    mname, task, seed = key
    tag = f"{mname}_{task}_seed{seed}"
    odir = f"{OUT_DIR}/{tag}"
    if os.path.exists(f"{odir}/DONE"):
        print("skip (done):", tag); return
    os.makedirs(odir, exist_ok=True)
    if (mname, task) not in cache:
        tok = AutoTokenizer.from_pretrained(path)
        cache[(mname, task)] = (tok, prep(task, tok), make_collate(tok))
    tok, data, collate = cache[(mname, task)]
    model = AutoModelForSequenceClassification.from_pretrained(path).to(dev_).eval()
    pr = Pruner(model)
    L, H = len(pr.acc), pr.H
    n_pool = len(data["dev"])
    nlist = [n for n in N_LIST if n <= n_pool]
    print(f"\n=== {tag}: L={L} H={H} dev pool={n_pool} val={len(data['val'])}  N_LIST={nlist}")

    W = weight_scores(model)
    G_ref, t_ref = grad_importance(model, data["dev_ref"], collate)
    A_ref, tA_ref = act_importance(model, data["dev_ref"], collate)
    ref_n = len(data["dev_ref"])
    print(f"  reference grad on {ref_n} ex: {t_ref:.1f}s | act on {ref_n} ex: {tA_ref:.1f}s | "
          f"tau(act, grad ref) = {mean_layer_tau(A_ref, G_ref):.3f}")

    rows, preds = [], {}

    def evaluate(method, n, draw, k, keep, extra):
        pr.apply(keep)
        lv, yv = predict(model, data["val"], collate)
        m = metrics(task, lv, yv)
        rows.append(dict(model=mname, task=task, seed=seed, method=method, n=n, draw=draw, k=k, n_kept=int(keep.sum()),
                         val_acc=m["acc"], val_loss=m["loss"], val_mcc=m["mcc"], val_f1=m["f1"], primary=m["primary"], **extra))
        preds[f"{method}|{n}|{draw}|{k}"] = lv.argmax(1).astype(np.int8)
        if "labels" not in preds: preds["labels"] = yv.astype(np.int8)

    evaluate("dense", 0, 0, H, np.ones((L, H), bool), {})
    dense = rows[0]["primary"]

    for k in K_LIST:
        ref_keep_g = uniform_keep(G_ref, k)
        # data-free references
        for name in ["frob", "op", "chain"]:
            keep = uniform_keep(W[name], k)
            evaluate(name, 0, 0, k, keep, dict(tau_to_ref=mean_layer_tau(W[name], G_ref), overlap_to_ref=overlap(keep, ref_keep_g), score_seconds=np.nan))
        for d in range(N_RANDOM):
            rs = np.random.RandomState(100000 * seed + 100 * k + d)
            keep = uniform_keep(rs.rand(L, H), k)
            evaluate("random", 0, d, k, keep, dict(tau_to_ref=np.nan, overlap_to_ref=overlap(keep, ref_keep_g), score_seconds=0.0))
        # reference gradient (n = ref_n) and reference act
        evaluate("grad", ref_n, -1, k, ref_keep_g, dict(tau_to_ref=1.0, overlap_to_ref=1.0, score_seconds=t_ref))
        keep = uniform_keep(A_ref, k)
        evaluate("act", ref_n, -1, k, keep, dict(tau_to_ref=mean_layer_tau(A_ref, G_ref), overlap_to_ref=overlap(keep, ref_keep_g), score_seconds=tA_ref))
    print(f"  references done (dense primary = {dense:.4f})")

    # subsets of the dev pool
    for n in nlist:
        for d in range(N_DRAWS):
            rs = np.random.RandomState(7919 * seed + 31 * n + d)
            idx = sorted(rs.choice(n_pool, n, replace=False).tolist())
            sub = data["dev"].select(idx)
            pr.apply(np.ones((L, H), bool))              # RESTORE the dense model: scores must come from the dense weights
            Gn, tg = grad_importance(model, sub, collate)
            An, ta = act_importance(model, sub, collate)
            tau_g, tau_a = mean_layer_tau(Gn, G_ref), mean_layer_tau(An, G_ref)
            for k in K_LIST:
                ref_keep_g = uniform_keep(G_ref, k)
                kg, ka = uniform_keep(Gn, k), uniform_keep(An, k)
                evaluate("grad", n, d, k, kg, dict(tau_to_ref=tau_g, overlap_to_ref=overlap(kg, ref_keep_g), score_seconds=tg))
                evaluate("act", n, d, k, ka, dict(tau_to_ref=tau_a, overlap_to_ref=overlap(ka, ref_keep_g), score_seconds=ta))
        print(f"  n={n} done")

    df = pd.DataFrame(rows)
    df["dense_primary"] = dense
    df["drop"] = dense - df["primary"]
    df.to_csv(f"{odir}/results.csv", index=False)
    np.savez_compressed(f"{odir}/val_preds.npz", **preds)
    np.savez_compressed(f"{odir}/scores.npz", G_ref=G_ref, A_ref=A_ref, **W)
    json.dump(dict(tag=tag, dense_primary=dense, N_LIST=nlist, N_DRAWS=N_DRAWS, K_LIST=K_LIST, ref_n=ref_n,
                   ref_grad_seconds=t_ref, ref_act_seconds=tA_ref), open(f"{odir}/meta.json", "w"), indent=2)
    open(f"{odir}/DONE", "w").write("ok")
    del model, pr
    torch.cuda.empty_cache()


def main():
    runs = discover()
    print(f"Found {len(runs)} checkpoints:")
    for k, v in runs.items():
        print("  ", k, "->", v)
    assert runs, "No checkpoints found: Add Input -> Notebook Output Files (fine-tuning notebooks)"
    cache = {}
    for key, path in runs.items():
        run_checkpoint(key, path, cache)
    allr = [pd.read_csv(p) for p in glob.glob(f"{OUT_DIR}/*/results.csv")]
    if allr:
        A = pd.concat(allr)
        A.to_csv(f"{OUT_DIR}/ALL_gradsamples_results.csv", index=False)
        s = A[A.method.isin(["grad", "act"])].groupby(["model", "task", "method", "n", "k"])["primary"].mean().unstack("k")
        print("\nMean primary metric by method / n / k:")
        print((s * 100).round(1).to_string())
    print("\nALL DONE ->", OUT_DIR)


if __name__ == "__main__" or True:
    main()

Found 18 checkpoints:
   ('bert-base-uncased', 'sst2', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed0
   ('bert-base-uncased', 'sst2', 1) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed1
   ('bert-base-uncased', 'sst2', 2) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed2
   ('distilbert-base-uncased', 'cola', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed0
   ('distilbert-base-uncased', 'cola', 1) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed1
   ('distilbert-base-uncased', 'cola', 2) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed2
   ('distilbert-base-uncased', 'cola', 3) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed3
   ('distilbert-base-uncased', 'cola', 4) -> /ka

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (63 outputs left out)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


=== distilbert-base-uncased_sst2_seed3: L=6 H=12 dev pool=3368 val=872  N_LIST=[8, 16, 32, 64, 128, 256]
  reference grad on 512 ex: 5.5s | act on 512 ex: 0.2s | tau(act, grad ref) = 0.662
  references done (dense primary = 0.9117)
  n=8 done
  n=16 done
  n=32 done
  n=64 done
  n=128 done
  n=256 done


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


=== distilbert-base-uncased_sst2_seed4: L=6 H=12 dev pool=3368 val=872  N_LIST=[8, 16, 32, 64, 128, 256]
  reference grad on 512 ex: 7.2s | act on 512 ex: 0.3s | tau(act, grad ref) = 0.707
  references done (dense primary = 0.9094)
  n=8 done
  n=16 done
  n=32 done
  n=64 done
  n=128 done
  n=256 done

Mean primary metric by method / n / k:
k                                           2     3     4     6
model                   task method n                          
bert-base-uncased       sst2 act    8    85.7  87.7  89.4  91.2
                                    16   84.6  87.5  89.4  91.3
                                    32   85.5  87.4  89.4  91.6
                                    64   85.5  87.1  89.3  91.4
                                    128  85.6  87.5  89.5  91.4
                                    256  86.0  87.6  89.4  91.4
                                    512  86.1  88.0  89.6  91.3
                             grad   8    78.6  84.6  87.3  91.0
              